# Step 2b: Fixing the Recovery Labels

Step 2 exposed four problems; this notebook handles each of them. **Parts A–C only read the files Step 2 already exported to Drive. No new exports are needed, and they run in a few minutes.**

| Part | Problem | Fix |
| --- | --- | --- |
| A | Weekly medians miss outages that end within a few days (Vamco, Noru) | For the first 4 weeks after landfall, use **nightly** data (median of 3 consecutive valid nights) to estimate outage duration in days |
| B | Kalmaegi dims broadly, even far from the track, which points to a baseline problem | Diagnose with the light trend of "far-field" municipalities for every typhoon; recompute Kalmaegi with an October 2025 baseline |
| C | Goni is cut short by Vamco 10 days later, so the two cannot be separated | Merge them into one compound event, `goni_vamco_2020` |
| D (optional) | Many rural municipalities are dropped for having too little light | Re-export with the minimum pixel count lowered from 8 to 4 and compare the noise (requires Earth Engine) |

The output is `step2b_labels.csv`: for each municipality, an **outage duration in days** (nightly estimate for short outages, weekly for long ones), a censoring flag, and whether it enters the modelling sample.

**Before running**: nothing to change. If you run Part D, replace `PROJECT_ID` there with your project ID.

In [ ]:
# 1. Install dependencies (Earth Engine is only used in Part D)
!pip install -q geopandas earthengine-api geemap

In [ ]:
# 2. Mount Google Drive (Step 2's exports live here)
import os, glob
from google.colab import drive
drive.mount("/content/drive")
DRIVE_FOLDER = "ph_typhoon_ntl"
DRIVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs("figures", exist_ok=True)
found = sorted(os.path.basename(f) for f in glob.glob(f"{DRIVE_DIR}/*.csv"))
print(len(found), "files:", found)

In [ ]:
# 3. Typhoon list and parameters (same as Step 2, plus the new Step 2b settings)
import numpy as np
import pandas as pd

EVENTS = pd.DataFrame([
    ("haiyan_2013",    "HAIYAN",     2013, "Yolanda", "2013-11-08 04:40", None,         "reference"),
    ("mangkhut_2018",  "MANGKHUT",   2018, "Ompong",  "2018-09-15 01:40", None,         "core"),
    ("kammuri_2019",   "KAMMURI",    2019, "Tisoy",   "2019-12-02 23:00", None,         "core"),
    ("phanfone_2019",  "PHANFONE",   2019, "Ursula",  "2019-12-24 16:45", "2019-11-28", "core"),
    ("goni_2020",      "GONI",       2020, "Rolly",   "2020-11-01 04:50", "2020-10-20", "core"),
    ("vamco_2020",     "VAMCO",      2020, "Ulysses", "2020-11-11 22:30", "2020-10-20", "core"),
    ("rai_2021",       "RAI",        2021, "Odette",  "2021-12-16 13:30", None,         "core"),
    ("noru_2022",      "NORU",       2022, "Karding", "2022-09-25 17:30", None,         "core"),
    ("manyi_2024",     "MAN-YI",     2024, "Pepito",  "2024-11-16 21:40", "2024-10-18", "optional"),
    ("kalmaegi_2025",  "KALMAEGI",   2025, "Tino",    "2025-11-04 00:00", "2025-09-20", "core"),
    ("fungwong_2025",  "FUNG-WONG",  2025, "Uwan",    "2025-11-09 21:10", "2025-09-20", "optional"),
], columns=["key", "ibtracs_name", "season", "local_name", "landfall", "baseline_end", "role"])
EVENTS["landfall"] = pd.to_datetime(EVENTS["landfall"])
EV = EVENTS.set_index("key")

# Step 2 parameters (unchanged)
BASE_DAYS, BASE_GAP_DAYS, POST_WEEKS = 120, 3, 26
MIN_ELIGIBLE_PX, MIN_VALID_FRAC, MOON_MAX_ILLUM = 8, 0.5, 0.8
MIN_DAYS_PER_WEEK, MIN_BASE_NIGHTS = 2, 8
OUTAGE_LEVEL = RECOVERY_LEVEL = 80
OVERPASS_LAG = pd.Timedelta(hours=25, minutes=30)

# New in 2b: nightly analysis
DAILY_DAYS = 28          # nightly analysis covers the first 28 days
ROLL_N = 3               # median of 3 consecutive valid nights
MAX_SPAN_DAYS = 6        # ...which must fall within 6 days
FIRST_LOOK_DAYS = 5      # first window must start within 5 days, or short outages cannot be judged
# Nightly analysis uses the same moon filter as the weekly one. Keeping bright-moon nights (1.0) was tested and
# rejected: under a bright moon the lunar correction can push radiance close to zero, which created false outages
# far from the track (Rai, >100 km: 0% -> 36%). Storms that made landfall near full moon (Rai, Kalmaegi) therefore
# lose their first week of nightly data and fall back on weekly medians
NIGHTLY_MOON_MAX_ILLUM = 0.8

# New in 2b: alternative Kalmaegi baseline (5 days after the 30 Sep northern Cebu quake, to 3 days before landfall)
KALMAEGI_ALT_BASE = ("2025-10-05", "2025-11-01")

def windows(ev):
    base_end = pd.Timestamp(ev.baseline_end) if isinstance(ev.baseline_end, str) else ev.landfall - pd.Timedelta(days=BASE_GAP_DAYS)
    return base_end - pd.Timedelta(days=BASE_DAYS), base_end

In [ ]:
# 4. Load typhoon tracks and municipal boundaries (used for maps only)
import requests
import geopandas as gpd
import matplotlib.pyplot as plt

IBTRACS_URL = ("https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/"
               "v04r01/access/csv/ibtracs.WP.list.v04r01.csv")
ib = pd.read_csv(IBTRACS_URL, usecols=["SID", "SEASON", "NAME", "ISO_TIME", "LAT", "LON"], skiprows=[1], low_memory=False)
ib["SEASON"] = pd.to_numeric(ib["SEASON"], errors="coerce")
ib = ib[ib["SEASON"] >= 2013].copy()
ib[["LAT", "LON"]] = ib[["LAT", "LON"]].apply(pd.to_numeric, errors="coerce")
ib["time_pht"] = pd.to_datetime(ib["ISO_TIME"]) + pd.Timedelta(hours=8)
tracks = {}
for ev in EVENTS.itertuples():
    t = ib[(ib.NAME == ev.ibtracs_name) & (ib.SEASON == ev.season)]
    if t.SID.nunique() > 1:
        t = t[t.SID == t.assign(g=(t.time_pht - ev.landfall).abs()).groupby("SID").g.min().idxmin()]
    tracks[ev.key] = t[t.LON.between(114, 130) & t.LAT.between(3, 22)].sort_values("time_pht")

def load_gb(level):
    try:
        meta = requests.get(f"https://www.geoboundaries.org/api/current/gbOpen/PHL/{level}/", timeout=60).json()
        return gpd.read_file(meta["simplifiedGeometryGeoJSON"])
    except Exception as e:
        print(f"{level}: API failed, trying the GitHub mirror ({e})")
        return gpd.read_file("https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/main/releaseData/"
                             f"gbOpen/PHL/{level}/geoBoundaries-PHL-{level}_simplified.geojson")

adm3 = load_gb("ADM3").to_crs(4326)[["shapeID", "geometry"]]
print(len(tracks), "tracks,", len(adm3), "municipalities")

INK, INK2, SURFACE, GRID = "#1f1f1e", "#5f5e5a", "#fcfcfb", "#e5e4e0"
plt.rcParams.update({"font.size": 10, "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.edgecolor": "#c9c8c4", "figure.facecolor": SURFACE,
                     "axes.facecolor": SURFACE})

In [ ]:
# 5. Label functions: weekly (as in Step 2) + nightly (new) + combined
REF_NEW_MOON, SYNODIC = pd.Timestamp("2000-01-06 18:14"), 29.530588853

def moon_illum(obs_utc):
    age = ((obs_utc - REF_NEW_MOON) / pd.Timedelta(days=1)) % SYNODIC
    return (1 - np.cos(2 * np.pi * age / SYNODIC)) / 2

def load_event(key, suffix=""):
    # read ntl_<key>.csv (and any shards Drive may create), never the _px4 files
    files = [f for f in sorted(glob.glob(f"{DRIVE_DIR}/ntl_{key}{suffix}*.csv"))
             if suffix or "_px4" not in os.path.basename(f)]
    if not files:
        return None
    d = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    d["date"] = pd.to_datetime(d["date"])
    d["obs_pht"] = d["date"] + OVERPASS_LAG
    d["illum"] = moon_illum(d["obs_pht"] - pd.Timedelta(hours=8))
    return d

def week_of(obs, t0):
    return (np.ceil((obs - t0) / pd.Timedelta(days=1) / 7) - 1).astype(int)

def label_all(g, t0, base_start, base_end, censor_at):
    # returns (label dict, weekly-median series)
    g_all = g
    ok_q = g["mean"].notna() & (g["count"] >= MIN_VALID_FRAC * g["eligible_px"])
    end = t0 + pd.Timedelta(weeks=POST_WEEKS)
    if pd.notna(censor_at):
        end = min(end, censor_at)

    # ---- weekly (bright-moon nights removed) ----
    g = g[ok_q & (g["illum"] <= MOON_MAX_ILLUM)].sort_values("obs_pht")
    base = g[(g.obs_pht >= base_start) & (g.obs_pht <= base_end)]
    out = {"base_nights": len(base)}
    if len(base) < 3:
        return {**out, "status_w": "no_baseline", "status_d": "no_baseline"}, None
    b = base["mean"].median()
    g = g.assign(pct=g["mean"] / b * 100, wk=week_of(g.obs_pht, t0))
    pre = g.loc[base.index].groupby("wk").pct.agg(["median", "size"])
    noise = pre.loc[pre["size"] >= MIN_DAYS_PER_WEEK, "median"].std()
    allw = g[g.obs_pht < end].groupby("wk").pct.agg(["median", "size"])
    allw = allw.loc[allw["size"] >= MIN_DAYS_PER_WEEK, "median"]
    w = allw[allw.index >= 0]
    out.update(baseline=round(b, 3), noise_w=round(noise, 1) if pd.notna(noise) else np.nan, post_weeks=len(w))
    for k in (1, 4, 12):
        near = w[(w.index >= k - 1) & (w.index <= k + 1)]
        out[f"pct_w{k}"] = round(near.iloc[np.argmin(np.abs(near.index - k))]) if len(near) else np.nan
    early = w[w.index <= 3]
    if early.empty:
        out["status_w"] = "no_early_data"
    else:
        low, low_wk = early.min(), early.idxmin()
        thr = OUTAGE_LEVEL if pd.isna(noise) else min(OUTAGE_LEVEL, 100 - 2 * noise)
        out.update(min_pct_w0_3=round(low), outage_w=bool(low < thr),
                   status_w="ok" if len(base) >= MIN_BASE_NIGHTS else "low_baseline")
        if not out["outage_w"]:
            out.update(recovery_weeks=0, censored_w=False)
        else:
            after = w[w.index >= low_wk]
            hit = [k for k, a, c in zip(after.index[:-1], after.values[:-1], after.values[1:])
                   if a >= RECOVERY_LEVEL and c >= RECOVERY_LEVEL]
            out.update(recovery_weeks=int(hit[0]) + 1 if hit else int(w.index.max()) + 1, censored_w=not hit)

    # ---- nightly: median of 3 consecutive valid nights (own moon setting) ----
    gn = g_all[ok_q & (g_all["illum"] <= NIGHTLY_MOON_MAX_ILLUM)].sort_values("obs_pht")
    base_n = gn[(gn.obs_pht >= base_start) & (gn.obs_pht <= base_end)]
    if len(base_n) < 3:
        return {**out, "status_d": "no_baseline"}, allw
    g = gn.assign(pct=gn["mean"] / base_n["mean"].median() * 100)
    bp = g.loc[base_n.index, "pct"].values
    base_rolls = [np.median(bp[i:i + ROLL_N]) for i in range(len(bp) - ROLL_N + 1)]
    noise3 = np.std(base_rolls, ddof=1) if len(base_rolls) > 2 else np.nan
    thr_d = OUTAGE_LEVEL if pd.isna(noise3) else min(OUTAGE_LEVEL, 100 - 2 * noise3)
    dend = min(end, t0 + pd.Timedelta(days=DAILY_DAYS))
    post = g[(g.obs_pht > t0) & (g.obs_pht < dend)]
    days = ((post.obs_pht - t0) / pd.Timedelta(days=1)).values
    vals = post.pct.values
    out.update(noise_3n=round(noise3, 1) if pd.notna(noise3) else np.nan, post_nights_28d=len(post),
               first_look_day=round(days[0], 1) if len(days) else np.nan)
    rolls = [(days[i], np.median(vals[i:i + ROLL_N])) for i in range(len(vals) - ROLL_N + 1)
             if days[i + ROLL_N - 1] - days[i] <= MAX_SPAN_DAYS]
    if not rolls:
        out["status_d"] = "too_few_nights"
    elif rolls[0][0] > FIRST_LOOK_DAYS:
        out["status_d"] = "late_first_look"
    else:
        rd, rv = np.array([r[0] for r in rolls]), np.array([r[1] for r in rolls])
        early_idx = np.where(rd <= FIRST_LOOK_DAYS)[0]
        i_min = early_idx[np.argmin(rv[early_idx])]
        out.update(min_pct_3n=round(rv[i_min]), outage_d=bool(rv[i_min] < thr_d), status_d="ok")
        if not out["outage_d"]:
            out.update(outage_days=0.0, censored_d=False)
        else:
            rec = np.where((rv >= RECOVERY_LEVEL) & (np.arange(len(rv)) > i_min))[0]
            out.update(outage_days=round(float(rd[rec[0]]), 1) if len(rec) else round(float(days[-1]), 1),
                       censored_d=not len(rec))
    return out, allw

OK_W = ("ok", "low_baseline")

def combine(r):
    # Combine both scales: an outage counts if either method finds one; short outages use nightly days, those beyond 28 days use weeks
    wk_ok = r.get("status_w") in OK_W
    d_ok = r.get("status_d") == "ok"
    wk_out = wk_ok and bool(r.get("outage_w"))
    d_out = d_ok and bool(r.get("outage_d"))
    if d_out and not r["censored_d"]:
        return r["outage_days"], False, "nightly"
    if d_out:                      # still out after 28 days on the nightly measure
        if wk_out:
            return max(DAILY_DAYS, r["recovery_weeks"] * 7), bool(r["censored_w"]), "weekly"
        return r["outage_days"], True, "nightly (censored)"
    if wk_out:                     # found by the weekly measure only
        return r["recovery_weeks"] * 7, bool(r["censored_w"]), "weekly"
    if d_ok or wk_ok:
        return 0.0, False, "nightly" if d_ok else "weekly"
    return np.nan, np.nan, "none"

In [ ]:
# 6. Define the event variants to compute, and label each one
def read_meta(key):
    m = pd.read_csv(f"{DRIVE_DIR}/meta_{key}.csv", parse_dates=["censor_at"])
    m["t0"] = EV.loc[key, "landfall"]
    return m

VARIANTS = []
for ev in EVENTS[EVENTS.role.isin(["reference", "core"])].itertuples():
    bs, be = windows(ev)
    VARIANTS.append(dict(name=ev.key, sources=[ev.key], base=(bs, be), baseline="step 2"))
VARIANTS.append(dict(name="kalmaegi_2025_octbase", sources=["kalmaegi_2025"],
                     base=tuple(pd.Timestamp(x) for x in KALMAEGI_ALT_BASE), baseline="October 2025"))
VARIANTS.append(dict(name="goni_vamco_2020", sources=["goni_2020", "vamco_2020"],
                     base=windows(EV.loc["goni_2020"].rename("goni_2020")), baseline="step 2"))
for extra in EVENTS[EVENTS.role == "optional"].itertuples():   # if Step 2 also ran the optional events
    if glob.glob(f"{DRIVE_DIR}/ntl_{extra.key}.csv"):
        VARIANTS.append(dict(name=extra.key, sources=[extra.key], base=windows(extra), baseline="step 2"))

rows, weekly_rows = [], []
for v in VARIANTS:
    data = [load_event(k) for k in v["sources"]]
    if any(d is None for d in data):
        print(f"{v['name']}: missing export file, skipped")
        continue
    metas = [read_meta(k).assign(src=k) for k in v["sources"]]
    if len(v["sources"]) == 1:
        d, meta = data[0], metas[0]
        meta["hit_by"] = v["sources"][0]
    else:
        # compound event: start from the earliest landfall that reached the municipality, no censoring, nearest track distance
        d = pd.concat(data, ignore_index=True).drop_duplicates(["shapeID", "date"])
        allm = pd.concat(metas, ignore_index=True)
        meta = (allm.sort_values("t0").groupby("shapeID")
                    .agg(muni=("muni", "first"), province=("province", "first"), dist_km=("dist_km", "min"),
                         eligible_px=("eligible_px", "max"), t0=("t0", "first"),
                         hit_by=("src", lambda s: "+".join(sorted(set(s)))))
                    .reset_index())
        meta["censor_at"] = pd.NaT
    d = d.merge(meta[["shapeID", "eligible_px"]], on="shapeID", how="left")
    info = meta.set_index("shapeID")
    for sid, g in d.groupby("shapeID"):
        m = info.loc[sid]
        r, allw = label_all(g, m.t0, *v["base"], m.censor_at)
        dur, cen, src = combine(r)
        rows.append({"variant": v["name"], "baseline_used": v["baseline"], "shapeID": sid, "muni": m.muni,
                     "province": m.province, "dist_km": m.dist_km, "eligible_px": m.eligible_px,
                     "hit_by": m.hit_by, "t0": m.t0, **r,
                     "duration_days": dur, "duration_censored": cen, "duration_source": src})
        if allw is not None:
            weekly_rows += [{"variant": v["name"], "shapeID": sid, "dist_km": m.dist_km, "wk": k, "pct": p}
                            for k, p in allw.items()]
    for m in meta[meta.eligible_px.fillna(0) < MIN_ELIGIBLE_PX].itertuples():
        rows.append({"variant": v["name"], "baseline_used": v["baseline"], "shapeID": m.shapeID, "muni": m.muni,
                     "province": m.province, "dist_km": m.dist_km, "eligible_px": m.eligible_px,
                     "hit_by": m.hit_by, "status_w": "too_few_lit_pixels", "status_d": "too_few_lit_pixels",
                     "duration_source": "none"})
    print(f"{v['name']}: {meta.shape[0]} municipalities")

labels = pd.DataFrame(rows)
weekly = pd.DataFrame(weekly_rows)
labels.groupby("variant", sort=False).status_d.value_counts().unstack(fill_value=0)

## A. Does the nightly analysis recover short outages?

Compare the share of municipalities flagged with an outage by each method, **within 30 km** of the track and at **100–150 km**:
- Near the track the share should rise (short outages recovered)
- Far from the track it should not rise much; otherwise the nightly method is "inventing" outages

`first_look_day` is the day on which the first usable 3-night window after landfall begins. If many municipalities exceed 5 days, clouds made short outages unobservable, which is a limitation worth reporting in its own right.

**About moonlight**: Rai (2021) and Kalmaegi (2025) made landfall around full moon, so dropping nights with lunar illumination above 80% removes their entire first week after landfall. Keeping those nights in the nightly analysis was tested and rejected: under a bright moon the lunar correction can push radiance close to zero, and Rai's outage share more than 100 km from the track jumped from 0% to 36%. Both analyses therefore use the 80% filter (`NIGHTLY_MOON_MAX_ILLUM = 0.8`), and storms that made landfall near full moon rely on weekly medians.

**Combination rule**: an outage counts if either method detects one. Outages up to 28 days use the nightly day count; longer ones are converted from weeks.

In [ ]:
# 7. Weekly vs nightly: share of municipalities with a detected outage
def method_table(x):
    lab = x[x.duration_source != "none"]
    near, far = lab[lab.dist_km <= 30], lab[lab.dist_km > 100]
    rec = lab[(lab.duration_days > 0) & (lab.duration_censored == False)]
    return pd.Series({
        "labelled": len(lab),
        "≤30 km weekly": near.outage_w.mean(),
        "≤30 km combined": (near.duration_days > 0).mean(),
        "100–150 km weekly": far.outage_w.mean(),
        "100–150 km combined": (far.duration_days > 0).mean(),
        "median first_look_day": lab.first_look_day.median(),
        "median outage days (recovered)": rec.duration_days.median(),
        "still out at window end": int((lab.duration_censored == True).sum()),
    })

A = labels.groupby("variant", sort=False).apply(method_table, include_groups=False)
A.round(2)

In [ ]:
# 8. Figure: detection share by method (near vs far)
C_WEEK, C_COMB = "#2a78d6", "#eb6834"
names = A.index.tolist()
y = np.arange(len(names))
fig, axes = plt.subplots(1, 2, figsize=(12, 0.45 * len(names) + 1.6), sharey=True, constrained_layout=True)
for ax, zone in zip(axes, ["≤30 km", "100–150 km"]):
    ax.barh(y - 0.2, A[f"{zone} weekly"] * 100, height=0.38, color=C_WEEK, label="Weekly medians (Step 2)")
    ax.barh(y + 0.2, A[f"{zone} combined"] * 100, height=0.38, color=C_COMB, label="Nightly + weekly (Step 2b)")
    ax.set_xlim(0, 100)
    ax.set_title(f"Municipalities {zone} from the track", loc="left", fontsize=11)
    ax.set_xlabel("Share with a detected outage (%)")
    ax.grid(axis="x", color=GRID, lw=0.8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes[0].set_yticks(y, names)
axes[0].invert_yaxis()
axes[1].legend(frameon=False, loc="lower right", fontsize=9)
fig.suptitle("Adding nightly analysis: more outages found near the track, few added far away",
             x=0.01, ha="left", fontsize=12)
fig.savefig("figures/step2b_method_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

## B. Baseline diagnostic: the light trend of far-field municipalities

Municipalities 100–150 km from the track are mostly only lightly affected. With a suitable baseline, their light should stay close to 100% throughout.
- Clearly off 100% before landfall: the baseline window and the pre-event period have different seasonal conditions
- Uniformly high or low after landfall, regardless of distance: more likely seasonal, holiday or other regional effects than the typhoon

In [ ]:
# 9. Figure: weekly median light of far-field municipalities (100–150 km) for each variant
far = weekly[(weekly.dist_km > 100) & weekly.wk.between(-12, 16)]
drift = far.groupby(["variant", "wk"]).pct.agg(median="median", n="size").reset_index()
vs = [v for v in names if v in set(drift.variant)]
ncol = 4
nrow = int(np.ceil(len(vs) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 2.9 * nrow), sharex=True, sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
for ax, v in zip(axes, vs):
    s = drift[(drift.variant == v) & (drift.n >= 5)]
    ax.axhspan(90, 110, color="#f0efec", zorder=0)
    ax.axhline(100, color="#a3a29d", lw=1, ls="--")
    ax.axvline(-0.5, color=INK2, lw=1)
    ax.plot(s.wk, s["median"], color=C_WEEK, lw=2, marker="o", ms=3)
    ax.set_ylim(40, 160)
    ax.set_title(v, loc="left", fontsize=10)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
for ax in axes[len(vs):]:
    ax.axis("off")
fig.supxlabel("Weeks relative to landfall (vertical line)", color=INK2)
fig.supylabel("Far-field light (% of baseline)", color=INK2)
fig.suptitle("Baseline check: municipalities 100–150 km from the track should stay near 100% (gray band = ±10%)",
             x=0.01, ha="left", fontsize=12)
fig.savefig("figures/step2b_baseline_drift.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 10. Kalmaegi: original baseline (May–Sep) vs October baseline, outage share by distance
BANDS = [0, 30, 60, 100, 150]
def by_band(x):
    lab = x[x.duration_source != "none"]
    return (lab.duration_days > 0).groupby(pd.cut(lab.dist_km, BANDS), observed=False).mean()

kal = {v: by_band(labels[labels.variant == v]) for v in ["kalmaegi_2025", "kalmaegi_2025_octbase"]
       if v in set(labels.variant)}
K = pd.DataFrame(kal).T
K.columns = [f"{int(c.left)}–{int(c.right)} km" for c in K.columns]
print((K * 100).round(0))

fig, ax = plt.subplots(figsize=(6.5, 3.8), constrained_layout=True)
x = np.arange(len(K.columns))
for (v, row), c, lab in zip(K.iterrows(), [C_WEEK, C_COMB], ["May–Sep baseline (Step 2)", "October baseline"]):
    ax.plot(x, row.values * 100, color=c, lw=2, marker="o", ms=6, label=lab)
ax.set_xticks(x, K.columns)
ax.set_ylim(0, 100)
ax.set_ylabel("Share with a detected outage (%)")
ax.set_xlabel("Distance to track")
ax.grid(axis="y", color=GRID, lw=0.8)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
ax.legend(frameon=False, fontsize=9)
ax.set_title("Kalmaegi (Tino) 2025: does a closer baseline restore the distance gradient?", loc="left", fontsize=11)
fig.savefig("figures/step2b_kalmaegi_baseline.png", dpi=200, bbox_inches="tight")
plt.show()

## C. The Goni–Vamco compound event

`goni_vamco_2020` merges the municipalities of both storms:
- Municipalities hit by both are followed for 26 weeks from Goni's landfall, with no cut-off (the `hit_by` column says which)
- Municipalities hit only by Vamco start from Vamco's landfall

For modelling, this compound event replaces `goni_2020` and `vamco_2020`.

In [ ]:
# 11. Comparing the three versions
A.loc[[v for v in ["goni_2020", "vamco_2020", "goni_vamco_2020"] if v in A.index]].round(2)

## Choose the modelling sample

The test for each storm: in the distance check, is the "high outage share near the track, low far away" gradient clear? Four storms pass and form the modelling sample: Mangkhut, Kammuri, Phanfone and Rai. The others are excluded:
- `goni_vamco_2020` and `noru_2022`: no gradient. These Luzon outages were short, and the first usable night often came after they had ended
- `kalmaegi_2025_octbase`: still flat even with the October baseline, which had a median of only 3 baseline nights
- Haiyan stays as the reference case only (no AlphaEarth embeddings before 2017)

In [ ]:
# 12. Choose the sample and save the final label table
MODEL_EVENTS = ["mangkhut_2018", "kammuri_2019", "phanfone_2019", "rai_2021"]
REFERENCE_EVENTS = ["haiyan_2013"]

final = labels[labels.variant.isin(MODEL_EVENTS + REFERENCE_EVENTS)].copy()
final["use_for_model"] = final.variant.isin(MODEL_EVENTS) & final.duration_days.notna()
final["event"] = final.variant.str.replace("_octbase", "", regex=False)
print(final.groupby("variant", sort=False).agg(
    municipalities=("shapeID", "size"),
    labelled=("duration_days", lambda s: s.notna().sum()),
    with_outage=("duration_days", lambda s: (s > 0).sum()),
    median_days=("duration_days", lambda s: s[s > 0].median()),
    for_model=("use_for_model", "sum")).to_string())

cols = ["event", "variant", "baseline_used", "shapeID", "muni", "province", "dist_km", "eligible_px", "hit_by",
        "duration_days", "duration_censored", "duration_source", "use_for_model",
        "outage_d", "outage_days", "censored_d", "min_pct_3n", "first_look_day", "noise_3n",
        "outage_w", "recovery_weeks", "censored_w", "min_pct_w0_3", "pct_w1", "pct_w4", "pct_w12", "noise_w",
        "base_nights", "status_w", "status_d"]
final = final[[c for c in cols if c in final.columns]]
final.to_csv("figures/step2b_labels.csv", index=False, encoding="utf-8-sig")
final.to_csv(f"{DRIVE_DIR}/step2b_labels.csv", index=False, encoding="utf-8-sig")
labels.to_csv(f"{DRIVE_DIR}/step2b_labels_all_variants.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 13. Distance check: outage share by distance band (final sample)
show = MODEL_EVENTS + REFERENCE_EVENTS
show = [v for v in show if v in set(final.variant)]
ncol = 4
nrow = int(np.ceil(len(show) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.8 * ncol, 2.8 * nrow), sharex=True, sharey=True, constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
xl = [f"{a}–{b}" for a, b in zip(BANDS[:-1], BANDS[1:])]
for ax, v in zip(axes, show):
    s = by_band(final[final.variant == v]) * 100
    n = final[(final.variant == v) & (final.duration_source != "none")].shape[0]
    ax.bar(range(len(s)), s.values, color="#2a78d6" if v in MODEL_EVENTS else "#a3a29d", width=0.7)
    ax.set_xticks(range(len(s)), xl, fontsize=8)
    ax.set_ylim(0, 100)
    ax.set_title(f"{v} (n = {n})", loc="left", fontsize=9)
    ax.grid(axis="y", color=GRID, lw=0.8)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
for ax in axes[len(show):]:
    ax.axis("off")
fig.supxlabel("Distance from municipality to track (km)", color=INK2)
fig.supylabel("Share with a detected outage (%)", color=INK2)
fig.suptitle("Closer to the track, more outages? Blue = modelling sample; gray = reference",
             x=0.01, ha="left", fontsize=12)
fig.savefig("figures/step2b_distance_check.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 14. Maps: outage duration (with fixed map extents)
from matplotlib.patches import Patch

CATS = [("No outage detected", "#cfcdc6"), ("≤ 3 days", "#86b6ef"), ("4–7 days", "#5598e7"),
        ("1–4 weeks", "#2a78d6"), ("1–4 months", "#1c5cab"), ("> 4 months or not yet", "#0d366b")]

def dur_cat(r):
    d, c = r.duration_days, r.duration_censored
    if pd.isna(d):
        return None
    if d == 0:
        return 0
    if c is True and d <= 112:
        return None
    return 1 if d <= 3 else 2 if d <= 7 else 3 if d <= 28 else 4 if d <= 112 else 5

ncol = 3
nrow = int(np.ceil(len(show) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 5 * nrow), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
for ax, v in zip(axes, show):
    lab = final[final.variant == v].copy()
    lab["cat"] = lab.apply(dur_cat, axis=1)
    geo = adm3.merge(lab[["shapeID", "cat", "duration_source"]], on="shapeID")
    measured = geo[geo.duration_source != "none"]
    pts = (measured if len(measured) else geo).representative_point()
    x0, x1 = np.quantile(pts.x, [0.02, 0.98]); y0, y1 = np.quantile(pts.y, [0.02, 0.98])
    pad = 0.4
    x0, x1, y0, y1 = x0 - pad, x1 + pad, y0 - pad, y1 + pad
    adm3.cx[x0:x1, y0:y1].plot(ax=ax, color="#f1f0ec", edgecolor="white", linewidth=0.2)
    unknown = geo[geo.cat.isna() & (geo.duration_source != "none")]
    if len(unknown):
        unknown.plot(ax=ax, facecolor="white", edgecolor="#a3a29d", hatch="////", linewidth=0.2)
    for i, (_, color) in enumerate(CATS):
        part = geo[geo.cat == i]
        if len(part):
            part.plot(ax=ax, color=color, edgecolor="white", linewidth=0.2)
    for k in ["goni_2020", "vamco_2020"] if v == "goni_vamco_2020" else [v.replace("_octbase", "")]:
        t = tracks.get(k)
        if t is not None:
            ax.plot(t.LON, t.LAT, color=INK, lw=1.2)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect("equal")
    ax.set_xlabel(""); ax.set_ylabel(""); ax.tick_params(labelsize=8)
    ax.set_title(v, loc="left", fontsize=11)
for ax in axes[len(show):]:
    ax.axis("off")
handles = [Patch(color=c, label=l) for l, c in CATS]
handles += [Patch(facecolor="white", edgecolor="#a3a29d", hatch="////", label="No call possible"),
            Patch(color="#f1f0ec", label="Too little light / outside swath")]
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.04))
fig.suptitle("How long municipal nighttime lights stayed below 80% of normal after each typhoon · NASA Black Marble",
             x=0.01, ha="left", fontsize=13)
fig.savefig("figures/step2b_duration_maps.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 15. Download results
from google.colab import files
for f in ["step2b_labels.csv", "step2b_method_comparison.png", "step2b_baseline_drift.png",
          "step2b_kalmaegi_baseline.png", "step2b_distance_check.png", "step2b_duration_maps.png"]:
    if os.path.exists(f"figures/{f}"):
        files.download(f"figures/{f}")

## D (optional): lower the light threshold from 8 pixels to 4

Re-export only the municipalities that had 4–7 eligible pixels in Step 2, then compare their noise and outage detection with the ≥ 8 pixel group:
- Similar noise and a clear distance gradient: the threshold can be relaxed, recovering more rural municipalities
- Clearly larger noise: keep the 8-pixel threshold and report "too little light" as a measurement limitation

By default this runs only the three events with the highest drop-out (Haiyan, Ursula, Odette). Exports take about 10–30 minutes.

In [ ]:
# 16. (optional) Re-export municipalities with 4–7 eligible pixels
import ee, geemap, time
PROJECT_ID = "ee-yourname"   # replace with your own Cloud project ID
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

PX4_EVENTS = ["haiyan_2013", "phanfone_2019", "rai_2021"]
MIN_PX_TEST = 4
BUILT_MIN_FRAC, MIN_RADIANCE = 0.03, 1.0     # same as Step 2

raw_col = ee.ImageCollection("NASA/VIIRS/002/VNP46A2")
proj = raw_col.first().select("DNB_BRDF_Corrected_NTL").projection()
def clean(img):
    ok = img.select("Mandatory_Quality_Flag").lte(1)
    cloud = img.select("QF_Cloud_Mask").rightShift(6).bitwiseAnd(3)
    return (img.select("DNB_BRDF_Corrected_NTL").updateMask(ok.And(cloud.lte(1))).rename("ntl")
               .copyProperties(img, ["system:time_start"]))
ntl = raw_col.map(clean)
settlement = (ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_S").filterDate("2020-01-01", "2021-01-01").first()
                .select("built_surface").divide(10000)
                .reduceResolution(ee.Reducer.mean(), maxPixels=64).reproject(proj)).gte(BUILT_MIN_FRAC)
REDUCER = ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True)

px4_tasks = {}
for key in PX4_EVENTS:
    ev = EV.loc[key].rename(key)
    meta = read_meta(key)
    ids = meta.loc[meta.eligible_px.between(MIN_PX_TEST, MIN_ELIGIBLE_PX - 1), "shapeID"].tolist()
    if not ids or glob.glob(f"{DRIVE_DIR}/ntl_{key}_px4*.csv"):
        print(key, "nothing to export (no matching municipalities, or file exists)")
        continue
    bs, be = windows(ev)
    pe = ev.landfall + pd.Timedelta(weeks=POST_WEEKS)
    base = (ntl.filterDate((bs - OVERPASS_LAG).strftime("%Y-%m-%d"), (be - OVERPASS_LAG).strftime("%Y-%m-%d"))
               .median().reproject(proj))
    eligible = settlement.And(base.gte(MIN_RADIANCE)).reproject(proj)
    fc = geemap.gdf_to_ee(adm3[adm3.shapeID.isin(ids)][["shapeID", "geometry"]])
    def per_image(img, eligible=eligible, fc=fc):
        date = img.date().format("YYYY-MM-dd")
        return img.updateMask(eligible).reduceRegions(fc, REDUCER, crs=proj).map(lambda f: f.set("date", date))
    i0 = ((bs - OVERPASS_LAG).normalize() - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    i1 = ((pe - OVERPASS_LAG).normalize() + pd.Timedelta(days=2)).strftime("%Y-%m-%d")
    daily = ntl.filterDate(i0, i1).map(per_image).flatten()
    task = ee.batch.Export.table.toDrive(collection=daily, description=f"ntl_{key}_px4", folder=DRIVE_FOLDER,
                                         fileNamePrefix=f"ntl_{key}_px4", fileFormat="CSV",
                                         selectors=["shapeID", "date", "mean", "count"])
    task.start()
    px4_tasks[key] = task
    print(f"{key}: {len(ids)} municipalities, export started")

while px4_tasks:
    states = {k: t.status()["state"] for k, t in px4_tasks.items()}
    print(time.strftime("%H:%M:%S"), states)
    if all(s in ("COMPLETED", "FAILED", "CANCELLED") for s in states.values()):
        break
    time.sleep(60)

In [ ]:
# 17. (optional) Compare the 4–7 pixel group with the ≥ 8 pixel group
cmp_rows = []
for key in PX4_EVENTS:
    d = load_event(key, suffix="_px4")
    if d is None:
        print(key, "no _px4 file yet")
        continue
    meta = read_meta(key)
    d = d.merge(meta[["shapeID", "eligible_px"]], on="shapeID", how="left")
    info = meta.set_index("shapeID")
    bs, be = windows(EV.loc[key].rename(key))
    for sid, g in d.groupby("shapeID"):
        m = info.loc[sid]
        r, _ = label_all(g, m.t0, bs, be, m.censor_at)
        dur, cen, src = combine(r)
        cmp_rows.append({"variant": key, "group": "4–7 px", "dist_km": m.dist_km, "noise_3n": r.get("noise_3n"),
                         "noise_w": r.get("noise_w"), "duration_days": dur, "duration_source": src})
    base8 = labels[(labels.variant == key) & (labels.duration_source != "none")]
    cmp_rows += [{"variant": key, "group": "≥8 px", "dist_km": r.dist_km, "noise_3n": r.noise_3n,
                  "noise_w": r.noise_w, "duration_days": r.duration_days, "duration_source": r.duration_source}
                 for r in base8.itertuples()]

if cmp_rows:
    C = pd.DataFrame(cmp_rows)
    C = C[C.duration_source != "none"]
    out = C.groupby(["variant", "group"]).apply(lambda x: pd.Series({
        "municipalities": len(x),
        "median nightly noise (±%)": x.noise_3n.median(),
        "median weekly noise (±%)": x.noise_w.median(),
        "outage share ≤30 km": (x[x.dist_km <= 30].duration_days > 0).mean(),
        "outage share 100–150 km": (x[x.dist_km > 100].duration_days > 0).mean(),
    }), include_groups=False)
    C.to_csv(f"{DRIVE_DIR}/step2b_px4_comparison.csv", index=False, encoding="utf-8-sig")
    display(out.round(2))